# External Validation — MLL23 Dataset

Interactive analysis of the WBC classifier's performance on the MLL23 external validation set.

**Run the pipeline first:**
```bash
python extension/external_validation/src/preprocess.py
python extension/external_validation/src/evaluate.py
python extension/external_validation/src/visualize.py
```

This notebook loads the outputs and adds statistical testing and domain shift quantification.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import Image, display
from scipy import stats

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

EXT_VAL_DIR = Path('..')
ROOT_DIR    = EXT_VAL_DIR.parent.parent

sys.path.append(str(ROOT_DIR / 'src'))
from dataset import CLASSES

RESULTS_DIR = EXT_VAL_DIR / 'results'
FIG_DIR     = RESULTS_DIR / 'figures'

## Load results

In [ ]:
df_clf     = pd.read_csv(RESULTS_DIR / 'classifications.csv')
df_metrics = pd.read_csv(RESULTS_DIR / 'metrics.csv', index_col=0)

cmp_path = RESULTS_DIR / 'bodzas_vs_mll23.csv'
df_cmp   = pd.read_csv(cmp_path, index_col=0) if cmp_path.exists() else None

overall_acc = df_clf['correct'].mean()

print(f'Total images evaluated : {len(df_clf)}')
print(f'Overall accuracy       : {overall_acc:.4f}')
print(f'Mean confidence        : {df_clf["confidence"].mean():.4f}')
print(f'Comparison available   : {df_cmp is not None}')

## Per-class performance

In [ ]:
per_cls = df_metrics.loc[[c for c in CLASSES if c in df_metrics.index],
                          ['precision', 'recall', 'f1-score', 'support']]

display(
    per_cls.style
    .background_gradient(subset=['precision', 'recall', 'f1-score'], cmap='RdYlGn', vmin=0, vmax=1)
    .format({'precision': '{:.4f}', 'recall': '{:.4f}', 'f1-score': '{:.4f}', 'support': '{:.0f}'})
)

## Bodzas vs MLL23 comparison

In [ ]:
if df_cmp is not None and 'bodzas_f1' in df_cmp.columns:
    show_cols = ['bodzas_f1', 'mll23_f1', 'f1_delta']
    df_show   = df_cmp.loc[[c for c in CLASSES if c in df_cmp.index], show_cols]

    display(
        df_show.style
        .background_gradient(subset=['bodzas_f1', 'mll23_f1'], cmap='RdYlGn', vmin=0, vmax=1)
        .background_gradient(subset=['f1_delta'], cmap='RdYlGn', vmin=-0.3, vmax=0.3)
        .format('{:.4f}')
    )

    mean_delta = df_show['f1_delta'].mean()
    print(f'Mean F1 delta (MLL23 − Bodzas): {mean_delta:+.4f}')
    degraded = df_show[df_show['f1_delta'] < -0.05]
    if len(degraded):
        print(f'Classes with >0.05 F1 drop: {list(degraded.index)}')
    else:
        print('No class degraded by more than 0.05 F1 points.')
else:
    print('Bodzas comparison not available — run 04_evaluate.ipynb first.')

## Domain shift quantification
Mean prediction confidence as a proxy for how much domain shift affects the model.
A large drop vs Bodzas test-set confidence indicates the model is less certain on MLL23.

In [ ]:
conf_summary = df_clf.groupby('true_class')['confidence'].agg(['mean', 'median', 'std'])
conf_summary = conf_summary.loc[[c for c in CLASSES if c in conf_summary.index]]

display(
    conf_summary.style
    .background_gradient(subset=['mean', 'median'], cmap='RdYlGn', vmin=0.5, vmax=1.0)
    .format('{:.4f}')
)

print(f'\nOverall mean confidence on MLL23: {df_clf["confidence"].mean():.4f}')
print('Compare this to the mean confidence on the Bodzas test set (from 04_evaluate.ipynb).')
print('A difference > 0.10 suggests meaningful domain shift.')

## Figures

In [ ]:
figures = [
    ('confusion_matrix.png',              'Confusion matrix'),
    ('per_class_f1.png',                  'Per-class F1'),
    ('bodzas_vs_mll23_f1.png',            'Bodzas vs MLL23 F1'),
    ('f1_delta.png',                      'F1 delta'),
    ('confidence_by_class.png',           'Confidence by class'),
    ('confidence_correct_vs_wrong.png',   'Confidence: correct vs incorrect'),
]

for filename, title in figures:
    path = FIG_DIR / filename
    if path.exists():
        print(f'\n{title}')
        display(Image(str(path)))
    else:
        print(f'[MISSING] {filename}')